# Handling Missing Data in Time Series

Missing data in time series is fundamentally different from cross-sectional data.
In a typical tabular dataset you might simply drop incomplete rows, but in time
series that is almost **never** the right move — it breaks the temporal structure,
destroys the regular frequency, and renders most downstream operations
(resampling, rolling windows, forecasting models) unusable.

Instead, we **fill** or **interpolate** the gaps. This notebook walks through
every major strategy, shows when each one is appropriate, and visualises the
results so you can judge quality at a glance.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

DATA_DIR = "../data/"

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

### Load the airline passengers dataset and create artificial gaps

In [ ]:
# Load the original clean data
df = pd.read_csv(
    DATA_DIR + "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
df.index.freq = pd.infer_freq(df.index)
df.columns = ["Passengers"]

print(f"Shape: {df.shape}")
print(f"Freq:  {df.index.freq}")
print(f"NaNs:  {df.isna().sum().values[0]}")
df.head()

In [ ]:
# Create a copy with artificial gaps
df_missing = df.copy()

# Scatter 15 random NaN values throughout the series
np.random.seed(42)
missing_idx = np.random.choice(df.index, size=15, replace=False)
df_missing.loc[missing_idx] = np.nan

# Also remove a contiguous 3-month block (Jun–Aug 1955)
df_missing.loc["1955-06":"1955-08"] = np.nan

print(f"Total NaN values: {df_missing.isna().sum().values[0]}")
df_missing.head(10)

---
## 1. Detecting Missing Data

In [ ]:
# Count missing values per column
print("Missing value counts:")
print(df_missing.isna().sum())
print()

# Boolean check — any missing at all?
print("Any missing values?")
print(df_missing.isna().any())

In [ ]:
# Which rows are missing?
missing_mask = df_missing["Passengers"].isna()
print(f"Missing dates ({missing_mask.sum()} total):")
print(df_missing.index[missing_mask].tolist())

### Visualise the gaps

Matplotlib naturally breaks the line wherever it encounters a `NaN`, so the gaps
are immediately visible.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: original vs gappy series
ax = axes[0]
ax.plot(df.index, df["Passengers"], color="steelblue", alpha=0.4, label="Original (complete)")
ax.plot(df_missing.index, df_missing["Passengers"], color="crimson", linewidth=1.5, label="With gaps")
ax.set_title("Airline Passengers — Original vs. Data with Artificial Gaps")
ax.set_ylabel("Thousands of Passengers")
ax.legend()

# Bottom: heatmap-style presence/absence matrix (msno-style)
ax = axes[1]
presence = df_missing["Passengers"].notna().astype(int).values.reshape(1, -1)
ax.imshow(presence, aspect="auto", cmap="RdYlGn", interpolation="nearest", vmin=0, vmax=1)
ax.set_yticks([])
# Label a handful of x-tick positions with dates
tick_positions = np.linspace(0, len(df_missing) - 1, 10, dtype=int)
ax.set_xticks(tick_positions)
ax.set_xticklabels([df_missing.index[i].strftime("%Y-%m") for i in tick_positions], rotation=45)
ax.set_title("Data Presence Map (green = present, red = missing)")

plt.tight_layout()
plt.show()

---
## 2. Why NOT to Drop Missing Values

Dropping rows is the default reflex from cross-sectional ML work, but for time
series it is almost always wrong.

In [ ]:
df_dropped = df_missing.dropna()

print(f"Original length:        {len(df)}")
print(f"After inserting gaps:    {len(df_missing)} (same length, but has NaNs)")
print(f"After dropna():         {len(df_dropped)}")
print()

# The frequency is now broken
inferred = pd.infer_freq(df_dropped.index)
print(f"Inferred frequency after dropna: {inferred}")
print()
print("Without a regular frequency, operations like resample(), rolling(),")
print("and most statsmodels functions will fail or produce wrong results.")

---
## 3. Forward Fill (`ffill`)

`ffill()` propagates the **last valid observation** forward into the gap.

**Best for:** price data, sensor readings, any series where "no news means the
old value still holds."

The `limit` parameter prevents filling through very long gaps where the carried
value would become unreasonable.

In [ ]:
df_ffill = df_missing.ffill()
df_ffill_limited = df_missing.ffill(limit=2)

print(f"NaNs after ffill():          {df_ffill.isna().sum().values[0]}")
print(f"NaNs after ffill(limit=2):   {df_ffill_limited.isna().sum().values[0]}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df.index, df["Passengers"], color="steelblue", alpha=0.35, label="Original")
ax.plot(df_missing.index, df_missing["Passengers"], "o", color="black", markersize=2, label="Observed (with gaps)")
ax.plot(df_ffill.index, df_ffill["Passengers"], color="darkorange", linewidth=1.2, label="ffill()")

# Highlight filled points
filled_mask = df_missing["Passengers"].isna()
ax.plot(
    df_ffill.index[filled_mask], df_ffill.loc[filled_mask, "Passengers"],
    "s", color="darkorange", markersize=6, label="Filled values",
)

ax.set_title("Forward Fill — Last Known Value Carried Forward")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Backward Fill (`bfill`)

`bfill()` propagates the **next valid observation** backward into the gap.

Less common than forward fill, but useful when you have future knowledge — for
example, filling in historical records from a known endpoint, or combining with
`ffill()` to close both ends of a gap.

In [ ]:
df_bfill = df_missing.bfill()

print(f"NaNs after bfill(): {df_bfill.isna().sum().values[0]}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df.index, df["Passengers"], color="steelblue", alpha=0.35, label="Original")
ax.plot(df_missing.index, df_missing["Passengers"], "o", color="black", markersize=2, label="Observed (with gaps)")
ax.plot(df_bfill.index, df_bfill["Passengers"], color="seagreen", linewidth=1.2, label="bfill()")

filled_mask = df_missing["Passengers"].isna()
ax.plot(
    df_bfill.index[filled_mask], df_bfill.loc[filled_mask, "Passengers"],
    "s", color="seagreen", markersize=6, label="Filled values",
)

ax.set_title("Backward Fill — Next Known Value Carried Backward")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Interpolation — The Best Approach for Most Time Series

Interpolation estimates missing values by fitting a curve through the known
points on either side of the gap. For smooth, continuous data this is almost
always superior to forward or backward fill.

### 5a. Linear interpolation (default)

Draws a straight line between the two known endpoints of each gap.

In [ ]:
df_linear = df_missing.interpolate(method="linear")

print(f"NaNs after linear interpolation: {df_linear.isna().sum().values[0]}")

### 5b. Time-weighted interpolation

`method='time'` uses the actual datetime index to weight the interpolation.
This matters when the time intervals between observations are unequal.

In [ ]:
df_time = df_missing.interpolate(method="time")

print(f"NaNs after time interpolation: {df_time.isna().sum().values[0]}")

### 5c. Spline interpolation

`method='spline'` fits a smooth polynomial spline through the data.
A cubic spline (`order=3`) handles curvature gracefully.

In [ ]:
df_spline = df_missing.interpolate(method="spline", order=3)

print(f"NaNs after spline interpolation: {df_spline.isna().sum().values[0]}")

### 5d. Polynomial interpolation

In [ ]:
df_poly = df_missing.interpolate(method="polynomial", order=2)

print(f"NaNs after polynomial interpolation: {df_poly.isna().sum().values[0]}")

### Compare linear vs. spline interpolation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

filled_mask = df_missing["Passengers"].isna()

for ax, (label, filled, color) in zip(axes, [
    ("Linear Interpolation", df_linear, "darkorange"),
    ("Cubic Spline Interpolation", df_spline, "purple"),
]):
    ax.plot(df.index, df["Passengers"], color="steelblue", alpha=0.35, label="Original")
    ax.plot(filled.index, filled["Passengers"], color=color, linewidth=1.2, label=label)
    ax.plot(
        filled.index[filled_mask], filled.loc[filled_mask, "Passengers"],
        "o", color=color, markersize=5, label="Interpolated points",
    )
    ax.set_ylabel("Thousands of Passengers")
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.show()

### All methods side by side

In [ ]:
methods = {
    "ffill()": df_ffill,
    "bfill()": df_bfill,
    "Linear": df_linear,
    "Time": df_time,
    "Spline (order=3)": df_spline,
    "Polynomial (order=2)": df_poly,
}

fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True, sharey=True)
colors = ["darkorange", "seagreen", "crimson", "teal", "purple", "saddlebrown"]

filled_mask = df_missing["Passengers"].isna()

for ax, (name, filled_df), color in zip(axes.flat, methods.items(), colors):
    ax.plot(df.index, df["Passengers"], color="steelblue", alpha=0.3, linewidth=0.8)
    ax.plot(filled_df.index, filled_df["Passengers"], color=color, linewidth=1)
    ax.plot(
        filled_df.index[filled_mask], filled_df.loc[filled_mask, "Passengers"],
        "o", color=color, markersize=4,
    )
    ax.set_title(name, fontsize=11)

fig.suptitle("Comparison of Missing-Data Strategies", fontsize=14, y=1.01)
fig.supylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

---
## 6. Choosing the Right Strategy

| Scenario | Best Method |
|----------|-------------|
| Price / stock data (last known value is best) | `ffill()` |
| Smooth continuous data (temperature, flow) | `interpolate(method='linear')` |
| Irregular time intervals | `interpolate(method='time')` |
| Data with curvature | `interpolate(method='spline', order=3)` |
| Strong seasonality with gaps | Seasonal interpolation or model-based imputation |
| Endpoints unknown | `bfill()` then `ffill()` or vice versa |
| Very long gaps | Consider keeping as NaN and handling in the model |

---
## 7. Resampling to Handle Irregular Data

If your timestamps are irregular (e.g., event-driven data), you can use
`resample()` or `asfreq()` to create a regular grid, then interpolate the gaps.

In [ ]:
# Simulate irregular timestamps by sampling random dates
np.random.seed(99)
irregular_dates = pd.to_datetime(
    np.sort(np.random.choice(pd.date_range("2020-01-01", "2020-12-31", freq="D"), size=50, replace=False))
)
irregular_series = pd.Series(
    np.cumsum(np.random.randn(50)) + 100,
    index=irregular_dates,
    name="value",
)

print(f"Irregular series length: {len(irregular_series)}")
print(f"Inferred freq: {pd.infer_freq(irregular_series.index)}")
print()
irregular_series.head(10)

In [ ]:
# Resample to daily frequency — this introduces NaNs on days without observations
regular_daily = irregular_series.asfreq("D")

print(f"Regular daily length: {len(regular_daily)}")
print(f"NaNs introduced:     {regular_daily.isna().sum()}")
print(f"Inferred freq:       {pd.infer_freq(regular_daily.index)}")

In [ ]:
# Fill the NaNs via interpolation
regular_filled = regular_daily.interpolate(method="time")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(regular_filled.index, regular_filled.values, color="steelblue", linewidth=1, label="Interpolated (daily)")
ax.plot(irregular_series.index, irregular_series.values, "o", color="crimson", markersize=5, label="Original irregular obs.")
ax.set_title("Irregular Timestamps Resampled to Daily Frequency + Time Interpolation")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

1. **Never drop** time series rows — it breaks temporal structure and frequency.
2. **Linear interpolation** (`interpolate(method='linear')`) is the safe default
   for most smooth, continuous data.
3. **Forward fill** (`ffill()`) is the standard for price / stock / last-known-value
   data.
4. **Spline / polynomial** interpolation can capture curvature but may overshoot
   in long gaps — always visualise.
5. For **irregular timestamps**, use `asfreq()` or `resample()` to establish a
   regular grid, then interpolate.
6. **Always visualise** the filled series against the original to verify that the
   imputation looks reasonable.